# LeetCode 2191 — Sort the Jumbled Numbers bằng Python

Link đề: https://leetcode.com/problems/sort-the-jumbled-numbers/ · Độ khó: **Medium** · Chủ đề: Array, Sorting

Skeleton LeetCode:

```python
class Solution:
    def sortJumbled(self, mapping: List[int], nums: List[int]) -> List[int]:
        ...
```

Sắp `nums` theo **giá trị sau khi đổi chữ số** (`mapping[d]` thay cho mỗi chữ số `d`), **ổn định** (bằng nhau thì giữ thứ tự gốc), và trả về **giá trị gốc**. Notebook đi từ **sắp chèn ổn định `O(n²)`** → **`sorted` với `cmp_to_key` tính lại giá trị mỗi lần so** → **`key` tính một lần bằng `str.translate`** → **`key` bằng số học** → **"trang trí–sắp–gỡ" (decorate–sort–undecorate) với chỉ số**.

## 1. Mục tiêu học

1. Tính **giá trị ánh xạ** của một số: thay từng chữ số, đọc lại như số thập phân (số 0 đầu tự mất).
2. Dùng tính **ổn định** của Timsort để giữ thứ tự gốc khi khóa bằng nhau.
3. Phân biệt `key=` (tính khóa **một lần** mỗi phần tử) với `cmp_to_key` (tính khi **mỗi lần so sánh**).
4. Xử lý cẩn thận trường hợp **số 0** (vòng `while x > 0` bỏ sót) và tránh lỗi "thay chuỗi dây chuyền" với `str.replace`.
5. Kiểm thử đối chiếu với sắp chèn ổn định viết tay.

## 2. Đề bài & ví dụ

`mapping` (độ dài 10, hoán vị của `0..9`): chữ số `i` được đổi thành `mapping[i]`. **Giá trị ánh xạ** của một số = số mới khi thay mọi chữ số `i` bằng `mapping[i]`. Trả về `nums` sắp **không giảm** theo giá trị ánh xạ.

- Phần tử có cùng giá trị ánh xạ giữ **thứ tự tương đối** như đầu vào.
- Chỉ **sắp theo** giá trị ánh xạ, **không thay** phần tử bằng giá trị đó.

**Ví dụ 1:** `mapping = [8,9,4,0,2,1,3,5,7,6]`, `nums = [991,338,38]` → `[338,38,991]` (`991 → 669`, `338 → 007 = 7`, `38 → 07 = 7`).

**Ví dụ 2:** `mapping = [0,1,...,9]`, `nums = [789,456,123]` → `[123,456,789]`.

**Ràng buộc:** `1 ≤ n ≤ 3·10⁴`, `0 ≤ nums[i] < 10⁹`, `mapping` là hoán vị của `0..9`.

In [1]:
import random
import time
from functools import cmp_to_key
from typing import List

examples = [([8, 9, 4, 0, 2, 1, 3, 5, 7, 6], [991, 338, 38], [338, 38, 991]),
            ([0, 1, 2, 3, 4, 5, 6, 7, 8, 9], [789, 456, 123], [123, 456, 789])]

def mapped_value(mapping, x):
    return int("".join(str(mapping[int(c)]) for c in str(x)))

for mp, nums, exp in examples:
    print({x: mapped_value(mp, x) for x in nums}, "-> kỳ vọng", exp)

{991: 669, 338: 7, 38: 7} -> kỳ vọng [338, 38, 991]
{789: 789, 456: 456, 123: 123} -> kỳ vọng [123, 456, 789]


## 3. Kiến thức nền

### 3.1 `str.maketrans` + `str.translate` — đổi nhiều ký tự **cùng lúc**

In [2]:
mp = [8, 9, 4, 0, 2, 1, 3, 5, 7, 6]
table = str.maketrans("0123456789", "".join(map(str, mp)))
print("991 ->", "991".translate(table), "| 338 ->", "338".translate(table), "->", int("338".translate(table)))

991 -> 669 | 338 -> 007 -> 7


### 3.2 `sorted(key=...)` ổn định

In [3]:
data = [("b", 2), ("a", 1), ("c", 2), ("d", 1)]
print(sorted(data, key=lambda p: p[1]), "<- cùng khóa: 'a' trước 'd', 'b' trước 'c' như đầu vào")

[('a', 1), ('d', 1), ('b', 2), ('c', 2)] <- cùng khóa: 'a' trước 'd', 'b' trước 'c' như đầu vào


### 3.3 Tách chữ số bằng số học

`x % 10` là chữ số hàng đơn vị, `x // 10` bỏ chữ số đó; dựng lại bằng `res += d · 10^k`. **Cẩn thận:** `x = 0` có một chữ số `0` — vòng `while x > 0` không chạy lần nào.

In [4]:
def digits_of(x):
    if x == 0:
        return [0]
    out = []
    while x:
        out.append(x % 10); x //= 10
    return out[::-1]

print(digits_of(9081), digits_of(0), digits_of(7))

[9, 0, 8, 1] [0] [7]


### 3.4 `cmp_to_key` — so sánh kiểu C

In [5]:
cmp = lambda a, b: (a > b) - (a < b)
print(sorted([3, 1, 2], key=cmp_to_key(cmp)))

[1, 2, 3]


## 4. Ý tưởng nền tảng / trực giác

Bài thực chất là `sorted(nums, key=f)` với `f(x)` = giá trị ánh xạ — vì `sorted` của Python **ổn định**, điều kiện "giữ thứ tự khi bằng nhau" tự thỏa.

Tính tay ví dụ 1 (`mapping[9] = 6`, `mapping[1] = 9`, `mapping[3] = 0`, `mapping[8] = 7`):

| `x` | chữ số | ánh xạ | giá trị |
|---|---|---|---|
| 991 | 9 9 1 | 6 6 9 | 669 |
| 338 | 3 3 8 | 0 0 7 | 7 |
| 38 | 3 8 | 0 7 | 7 |

Sắp theo giá trị: `7 (338), 7 (38), 669 (991)` ⇒ `[338, 38, 991]` ✓ (`338` đứng trước `38` vì vậy ở đầu vào).

Lưu ý: so sánh **giá trị số**, không phải chuỗi `"007"` vs `"07"` — số 0 đầu không quan trọng.

## 5. Cách 1 — Brute Force / baseline: sắp chèn ổn định viết tay

Chèn từng phần tử vào đúng chỗ trong phần đã sắp, chỉ dịch qua phần tử có khóa **lớn hơn hẳn** (không vượt qua khóa bằng nhau ⇒ ổn định).

In [6]:
def jumbled_insertion(mapping, nums):
    keys = [mapped_value(mapping, x) for x in nums]
    order = []                                   # danh sách chỉ số đã sắp
    for i in range(len(nums)):
        j = len(order)
        while j > 0 and keys[order[j - 1]] > keys[i]:   # '>' nghiêm ngặt -> ổn định
            j -= 1
        order.insert(j, i)
    return [nums[i] for i in order]

for mp, nums, exp in examples:
    print(jumbled_insertion(mp, nums), "(kỳ vọng", exp, ")")

[338, 38, 991] (kỳ vọng [338, 38, 991] )
[123, 456, 789] (kỳ vọng [123, 456, 789] )


**Độ phức tạp:** `O(n²)` (chèn vào list) — `9·10⁸` với `n = 3·10⁴`: quá chậm, nhưng rất dễ thấy đúng ⇒ chuẩn đối chiếu. **Nút thắt:** sắp xếp so sánh `O(n log n)` của thư viện + khóa tính sẵn.

## 6. Ý tưởng thuật toán / bước đột phá

**Insight:** khóa sắp xếp là hàm của **riêng** từng phần tử ⇒ tính **một lần** (`key=`), rồi để Timsort ổn định lo phần còn lại.

**Quy tắc tính khóa:**
- **nếu** dùng chuỗi: `int(str(x).translate(table))` — `translate` đổi mọi chữ số **đồng thời**, `int` bỏ số 0 đầu;
- **nếu** dùng số học: duyệt chữ số từ hàng đơn vị, `res += mapping[d] · 10^k`; **xử lý riêng** `x = 0` ⇒ `mapping[0]`.

Không cần tạo cặp `(khóa, chỉ số)` vì `sorted` đã ổn định; nhưng ở ngôn ngữ dùng sắp xếp **không** ổn định (C++ `std::sort`), phải thêm chỉ số vào khóa.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — `cmp_to_key`, tính lại giá trị ánh xạ mỗi lần so sánh

In [7]:
def jumbled_cmp(mapping, nums):
    def cmp(a, b):
        ma, mb = mapped_value(mapping, a), mapped_value(mapping, b)
        return (ma > mb) - (ma < mb)
    return sorted(nums, key=cmp_to_key(cmp))

for mp, nums, exp in examples:
    print(jumbled_cmp(mp, nums), "(kỳ vọng", exp, ")")

[338, 38, 991] (kỳ vọng [338, 38, 991] )
[123, 456, 789] (kỳ vọng [123, 456, 789] )


**Độ phức tạp:** `O(n log n · d)` với `d ≤ 9` chữ số, nhưng hằng số lớn (mỗi phép so tính lại **hai** khóa). Vẫn ổn định (Timsort).

### 7.2 Cách 3 — `key=` bằng `str.translate`

In [8]:
def jumbled_translate(mapping, nums):
    table = str.maketrans("0123456789", "".join(map(str, mapping)))
    return sorted(nums, key=lambda x: int(str(x).translate(table)))

for mp, nums, exp in examples:
    print(jumbled_translate(mp, nums), "(kỳ vọng", exp, ")")

[338, 38, 991] (kỳ vọng [338, 38, 991] )
[123, 456, 789] (kỳ vọng [123, 456, 789] )


**Độ phức tạp:** `O(n · d + n log n)` — khóa tính một lần, `translate` chạy trong C.

### 7.3 Cách 4 — `key=` bằng số học

In [9]:
def mapped_arith(mapping, x):
    if x == 0:
        return mapping[0]
    res, p = 0, 1
    while x:
        res += mapping[x % 10] * p
        p *= 10
        x //= 10
    return res

def jumbled_arith(mapping, nums):
    return sorted(nums, key=lambda x: mapped_arith(mapping, x))

for mp, nums, exp in examples:
    print(jumbled_arith(mp, nums), "(kỳ vọng", exp, ")")

[338, 38, 991] (kỳ vọng [338, 38, 991] )
[123, 456, 789] (kỳ vọng [123, 456, 789] )


**Vì sao cộng `mapping[d] · p` lại tự "bỏ số 0 đầu"?** Chữ số đầu (hàng cao nhất) ánh xạ thành `0` chỉ đóng góp `0` — giá trị số không đổi, đúng như đọc `"007"` thành `7`.

### 7.4 Cách 5 — Trang trí–sắp–gỡ (decorate–sort–undecorate) với chỉ số

Tạo cặp `(khóa, chỉ số, giá trị)`, sắp tuple (so khóa trước, rồi chỉ số ⇒ ổn định **tường minh**), rồi lấy giá trị. Cách viết chuẩn cho ngôn ngữ có sắp xếp không ổn định.

In [10]:
def jumbled_dsu(mapping, nums):
    decorated = [(mapped_arith(mapping, x), i, x) for i, x in enumerate(nums)]
    decorated.sort()
    return [x for _, _, x in decorated]

for mp, nums, exp in examples:
    print(jumbled_dsu(mp, nums), "(kỳ vọng", exp, ")")

[338, 38, 991] (kỳ vọng [338, 38, 991] )
[123, 456, 789] (kỳ vọng [123, 456, 789] )


## 8. TRACE từng bước

### 8.1 Trace tính khóa bằng số học cho ví dụ 1

In [11]:
def trace_arith(mapping, x):
    if x == 0:
        print(f"    x = 0 -> mapping[0] = {mapping[0]}"); return mapping[0]
    res, p, orig = 0, 1, x
    while x:
        d = x % 10
        res += mapping[d] * p
        print(f"    chữ số {d} (hàng {p}) -> mapping = {mapping[d]}, cộng {mapping[d] * p}, res = {res}")
        p *= 10; x //= 10
    print(f"  {orig} -> {res}")
    return res

mp1 = examples[0][0]
for x in [991, 338, 38]:
    trace_arith(mp1, x)

    chữ số 1 (hàng 1) -> mapping = 9, cộng 9, res = 9
    chữ số 9 (hàng 10) -> mapping = 6, cộng 60, res = 69
    chữ số 9 (hàng 100) -> mapping = 6, cộng 600, res = 669
  991 -> 669
    chữ số 8 (hàng 1) -> mapping = 7, cộng 7, res = 7
    chữ số 3 (hàng 10) -> mapping = 0, cộng 0, res = 7
    chữ số 3 (hàng 100) -> mapping = 0, cộng 0, res = 7
  338 -> 7
    chữ số 8 (hàng 1) -> mapping = 7, cộng 7, res = 7
    chữ số 3 (hàng 10) -> mapping = 0, cộng 0, res = 7
  38 -> 7


### 8.2 Trace sắp chèn ổn định trên dữ liệu tự dựng có nhiều khóa trùng (đã đối chiếu `sorted` — đối chiếu lại đề gốc)

In [12]:
def trace_insertion(mapping, nums):
    keys = [mapped_value(mapping, x) for x in nums]; order = []
    for i in range(len(nums)):
        j = len(order)
        while j > 0 and keys[order[j - 1]] > keys[i]:
            j -= 1
        order.insert(j, i)
        print(f"    chèn {nums[i]} (khóa {keys[i]}) vào vị trí {j}: {[nums[k] for k in order]}")
    return [nums[k] for k in order]

mp_demo = [9, 8, 7, 6, 5, 4, 3, 2, 1, 0]       # đảo ngược chữ số: d -> 9 - d
nums_demo = [0, 90, 9, 99, 1, 8]
r = trace_insertion(mp_demo, nums_demo)
print("  =>", r, "| sorted:", jumbled_translate(mp_demo, nums_demo))

    chèn 0 (khóa 9) vào vị trí 0: [0]
    chèn 90 (khóa 9) vào vị trí 1: [0, 90]
    chèn 9 (khóa 0) vào vị trí 0: [9, 0, 90]
    chèn 99 (khóa 0) vào vị trí 1: [9, 99, 0, 90]
    chèn 1 (khóa 8) vào vị trí 2: [9, 99, 1, 0, 90]
    chèn 8 (khóa 1) vào vị trí 2: [9, 99, 8, 1, 0, 90]
  => [9, 99, 8, 1, 0, 90] | sorted: [9, 99, 8, 1, 0, 90]


## 9. Code nộp LeetCode

In [13]:
class Solution:
    def sortJumbled(self, mapping: List[int], nums: List[int]) -> List[int]:
        def mapped(x: int) -> int:
            if x == 0:
                return mapping[0]
            res, p = 0, 1
            while x:
                res += mapping[x % 10] * p
                p *= 10
                x //= 10
            return res
        return sorted(nums, key=mapped)           # Timsort ổn định: khóa bằng nhau giữ thứ tự gốc

for mp, nums, exp in examples:
    assert Solution().sortJumbled(mp, nums) == exp
print("Ví dụ đề: OK")

Ví dụ đề: OK


## 10. Vì sao thuật toán đúng

1. **Khóa đúng:** với `x > 0` có chữ số `d_{L−1} … d_1 d_0`, giá trị ánh xạ là `Σ mapping[d_k] · 10^k` — đúng cách đọc dãy chữ số mới như số thập phân (số 0 đầu đóng góp 0). Với `x = 0`, dãy chữ số là `[0]` ⇒ `mapping[0]`.
2. **Thứ tự:** `sorted(..., key=f)` trả về hoán vị của `nums` mà `f` không giảm.
3. **Ổn định:** Timsort giữ thứ tự gốc của các phần tử có `f` bằng nhau — đúng yêu cầu "same relative order".
4. **Không thay giá trị:** `sorted` trả về phần tử gốc, khóa chỉ dùng để so sánh.
5. **Cách 5** ổn định tường minh: hai phần tử khóa bằng nhau so tiếp theo chỉ số gốc.

## 11. Phân tích Time / Space Complexity

Gọi `n = len(nums)`, `d ≤ 9` = số chữ số.

| Cách | Time | Space |
|---|---|---|
| 1. Sắp chèn | `O(n² + n d)` | `O(n)` |
| 2. `cmp_to_key` | `O(n log n · d)` (hằng số lớn) | `O(n)` |
| 3. `translate` | `O(n d + n log n)` | `O(n)` |
| 4. Số học | `O(n d + n log n)` | `O(n)` |
| 5. Decorate–sort–undecorate | `O(n d + n log n)` | `O(n)` |

In [14]:
rnd = random.Random(2191)
mp_big = list(range(10)); rnd.shuffle(mp_big)
nums_big = [rnd.randrange(10**9) for _ in range(3 * 10**4)]
for name, fn in [("cmp", jumbled_cmp), ("translate", jumbled_translate), ("arith", jumbled_arith),
                 ("dsu", jumbled_dsu), ("Solution", Solution().sortJumbled)]:
    t0 = time.perf_counter(); r = fn(mp_big, nums_big); dt = time.perf_counter() - t0
    print(f"{name:<10}: {dt * 1000:7.1f} ms")
t0 = time.perf_counter(); jumbled_insertion(mp_big, nums_big[:3000]); print(f"insertion (n=3000): {(time.perf_counter() - t0) * 1000:7.1f} ms")

cmp       :   989.8 ms
translate :    11.3 ms
arith     :    26.7 ms
dsu       :    31.5 ms
Solution  :    21.6 ms


insertion (n=3000):   124.4 ms


## 12. Bộ test

In [15]:
ALL = {
    "cmp": jumbled_cmp,
    "translate": jumbled_translate,
    "arith": jumbled_arith,
    "dsu": jumbled_dsu,
    "Solution": lambda mp, a: Solution().sortJumbled(mp, a),
}

def is_valid(mapping, nums, out):
    """Validator: hoán vị của nums, khóa không giảm, cùng khóa giữ thứ tự gốc."""
    if sorted(out) != sorted(nums):
        return False
    keys = [mapped_value(mapping, x) for x in out]
    if any(keys[i] > keys[i + 1] for i in range(len(keys) - 1)):
        return False
    from collections import defaultdict
    g_out, g_in = defaultdict(list), defaultdict(list)     # nhóm theo khóa, một lượt mỗi bên
    for x, k in zip(out, keys):
        g_out[k].append(x)
    for x in nums:
        g_in[mapped_value(mapping, x)].append(x)
    return g_out == g_in

def check(mapping, nums, exp=None, brute=True):
    ref = jumbled_insertion(mapping, nums) if brute else jumbled_translate(mapping, nums)
    assert is_valid(mapping, nums, ref)
    for name, fn in ALL.items():
        got = fn(mapping, list(nums))
        assert got == ref, (name, mapping, nums, got, ref)
    if exp is not None:
        assert ref == exp
    return ref

for mp, nums, exp in examples:
    check(mp, nums, exp)
print("Ví dụ đề: OK")

ident = list(range(10))
check(ident, [0], [0])
check([9, 8, 7, 6, 5, 4, 3, 2, 1, 0], [0, 9], [9, 0])           # 0 -> 9, 9 -> 0
check([5, 0, 1, 2, 3, 4, 6, 7, 8, 9], [10, 1, 0])               # khóa: 10->05=5, 1->0, 0->5
check(ident, [5, 5, 5, 1], [1, 5, 5, 5])
check([0, 2, 1, 3, 4, 5, 6, 7, 8, 9], [12, 21, 11, 22])           # đổi vai 1 và 2
print("Edge case (có số 0, khóa trùng — tự dựng, đối chiếu sắp chèn): OK")

rnd = random.Random(2021)
for _ in range(1500):
    mp = list(range(10)); rnd.shuffle(mp)
    n = rnd.randint(1, 15)
    nums = [rnd.choice([0, rnd.randrange(10), rnd.randrange(100), rnd.randrange(10**9)]) for _ in range(n)]
    check(mp, nums)
print("1500 bộ ngẫu nhiên so với sắp chèn: OK")

for _ in range(5):
    mp = list(range(10)); rnd.shuffle(mp)
    check(mp, [rnd.randrange(10**9) for _ in range(3 * 10**4)], brute=False)
print("5 bộ lớn n = 3·10⁴ (đối chiếu chéo + validator): OK")

Ví dụ đề: OK
Edge case (có số 0, khóa trùng — tự dựng, đối chiếu sắp chèn): OK
1500 bộ ngẫu nhiên so với sắp chèn: OK


5 bộ lớn n = 3·10⁴ (đối chiếu chéo + validator): OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Số học nhưng quên trường hợp `x = 0`

In [16]:
def bug_zero(mapping, nums):
    def f(x):
        res, p = 0, 1
        while x:                                   # SAI: x = 0 cho res = 0 thay vì mapping[0]
            res += mapping[x % 10] * p; p *= 10; x //= 10
        return res
    return sorted(nums, key=f)

mp, nums = [9, 8, 7, 6, 5, 4, 3, 2, 1, 0], [0, 1]
print("sai :", bug_zero(mp, nums), "| đúng:", Solution().sortJumbled(mp, nums), "(0 -> 9, 1 -> 8)")

sai : [0, 1] | đúng: [1, 0] (0 -> 9, 1 -> 8)


**Sửa:** `if x == 0: return mapping[0]`.

### Lỗi 2 — Thay chữ số lần lượt bằng `str.replace` (thay dây chuyền)

In [17]:
def chain_key(mapping, x):
    s = str(x)
    for d in range(10):
        s = s.replace(str(d), str(mapping[d]))         # SAI: chữ số vừa thay có thể bị thay tiếp
    return int(s)

def bug_chain_replace(mapping, nums):
    return sorted(nums, key=lambda x: chain_key(mapping, x))

mp, nums = [1, 0, 2, 3, 4, 5, 6, 7, 8, 9], [10, 1, 0]
print("khóa sai:", [chain_key(mp, x) for x in nums], "| khóa đúng:", [mapped_value(mp, x) for x in nums])
print("sai :", bug_chain_replace(mp, nums), "| đúng:", Solution().sortJumbled(mp, nums))

khóa sai: [0, 0, 0] | khóa đúng: [1, 0, 1]
sai : [10, 1, 0] | đúng: [1, 10, 0]


**Vì sao sai:** với `mapping = [1, 0, …]`, bước `d = 0` đổi `0 → 1`, rồi bước `d = 1` đổi **mọi** `1` (kể cả `1` vừa tạo) thành `0`. **Sửa:** `str.translate` (đổi đồng thời) hoặc dựng chuỗi mới từng ký tự.

### Lỗi 3 — So sánh **chuỗi** ánh xạ (thứ tự từ điển, giữ số 0 đầu)

In [18]:
def bug_string_key(mapping, nums):
    table = str.maketrans("0123456789", "".join(map(str, mapping)))
    return sorted(nums, key=lambda x: str(x).translate(table))     # SAI: so sánh chuỗi, không phải số

mp, nums = list(range(10)), [9, 10]
print("sai :", bug_string_key(mp, nums), "| đúng:", Solution().sortJumbled(mp, nums))
mp1, nums1 = examples[0][0], examples[0][1]
print("ví dụ 1 -> sai:", bug_string_key(mp1, nums1), "| đúng:", Solution().sortJumbled(mp1, nums1))
print("[38, 338] -> sai:", bug_string_key(mp1, [38, 338]), "| đúng:", Solution().sortJumbled(mp1, [38, 338]))

sai : [10, 9] | đúng: [9, 10]
ví dụ 1 -> sai: [338, 38, 991] | đúng: [338, 38, 991]
[38, 338] -> sai: [338, 38] | đúng: [38, 338]


**Vì sao sai:** `"10" < "9"` theo từ điển. Ví dụ 1 lại **tình cờ** đúng (`"007" < "07" < "669"`), nhưng `"007"` và `"07"` lẽ ra phải **bằng nhau** (cùng giá trị 7) — nếu thứ tự đầu vào là `[38, 338]` thì bản lỗi sẽ đảo chúng, phá tính ổn định. **Sửa:** `int(...)`.

### Lỗi 4 — Trả về giá trị **đã ánh xạ** thay vì giá trị gốc

In [19]:
def bug_return_mapped(mapping, nums):
    return sorted(mapped_value(mapping, x) for x in nums)           # SAI

mp1, nums1 = examples[0][0], examples[0][1]
print("sai :", bug_return_mapped(mp1, nums1), "| đúng:", Solution().sortJumbled(mp1, nums1))

sai : [7, 7, 669] | đúng: [338, 38, 991]


**Sửa:** sắp `nums` với `key=`, không thay phần tử.

### Lỗi 5 — Phá hòa bằng giá trị gốc (mất tính ổn định)

In [20]:
def bug_tiebreak_value(mapping, nums):
    return sorted(nums, key=lambda x: (mapped_value(mapping, x), x))   # SAI: khóa bằng nhau phải giữ thứ tự đầu vào

mp1, nums1 = examples[0][0], examples[0][1]
print("sai :", bug_tiebreak_value(mp1, nums1), "| đúng:", Solution().sortJumbled(mp1, nums1))

sai : [38, 338, 991] | đúng: [338, 38, 991]


**Vì sao sai:** `338` và `38` cùng khóa 7; phá hòa theo giá trị đưa `38` lên trước. Đề yêu cầu giữ thứ tự gốc ⇒ phá hòa (nếu cần) phải theo **chỉ số**, không theo giá trị.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Sắp theo một hàm của phần tử, bằng nhau giữ thứ tự" ⇒ `sorted(nums, key=f)` (Python ổn định).
- Đổi nhiều ký tự cùng lúc ⇒ `str.maketrans` + `translate`; **tránh** chuỗi `replace`.
- Số học: nhớ `x = 0`.
- Ngôn ngữ có sắp không ổn định ⇒ khóa `(f(x), index)`.
- Test có: số `0`, khóa trùng, mapping làm chữ số đầu thành 0.

## 15. Biến thể của bài

1. **LeetCode 1356 — Sort Integers by The Number of 1 Bits:** khóa `(popcount, x)` (ở đó phá hòa theo **giá trị**).
2. **LeetCode 1636 — Sort Array by Increasing Frequency:** khóa `(freq, −x)`.
3. **LeetCode 791 — Custom Sort String:** thứ tự ký tự tùy biến ⇒ khóa = vị trí trong `order`.
4. **Thứ tự chữ số tùy biến theo kiểu từ điển** (so chuỗi theo bảng chữ cái mới) — khác bài này (so giá trị số).
5. **Ánh xạ không là hoán vị** (nhiều chữ số cùng ánh xạ một giá trị) — thuật toán không đổi; chỉ là nhiều khóa trùng hơn.

In [21]:
def custom_sort_string_791(order, s):
    pos = {c: i for i, c in enumerate(order)}
    return "".join(sorted(s, key=lambda c: pos.get(c, len(order))))

def sort_by_bits_1356(arr):
    return sorted(arr, key=lambda x: (bin(x).count("1"), x))

print("791:", custom_sort_string_791("cba", "abcd"), "| 1356:", sort_by_bits_1356([0, 1, 2, 3, 4, 5, 6, 7, 8]))
mp_dup = [0, 0, 1, 1, 2, 2, 3, 3, 4, 4]                       # biến thể 5: không phải hoán vị
print("ánh xạ không hoán vị:", Solution().sortJumbled(mp_dup, [23, 32, 1, 0, 10]), "khóa", [mapped_value(mp_dup, x) for x in [23, 32, 1, 0, 10]])

791: cbad | 1356: [0, 1, 2, 4, 8, 3, 5, 6, 7]
ánh xạ không hoán vị: [1, 0, 10, 23, 32] khóa [11, 11, 0, 0, 0]


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Sắp chèn | ổn định thủ công | `O(n²)` | `O(n)` | kiểm thử |
| 2. `cmp_to_key` | tính lại khóa mỗi lần so | `O(n log n d)` | `O(n)` | khi chỉ có hàm so sánh |
| 3. `translate` | khóa chuỗi → số | `O(n d + n log n)` | `O(n)` | Python gọn |
| 4. Số học | `Σ mapping[d]·10^k` | `O(n d + n log n)` | `O(n)` | **nộp bài** |
| 5. Decorate–sort–undecorate | `(khóa, chỉ số)` | `O(n d + n log n)` | `O(n)` | sắp không ổn định |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** `Solution().sortJumbled([9,8,7,6,5,4,3,2,1,0], [0, 1, 2, 10, 99])`?

**Bài 2 (tính tay):** Với `mapping = [3,0,1,2,4,5,6,7,8,9]`, tính giá trị ánh xạ của `100`, `3`, `20`, `0`.

**Bài 3 (điền TODO):** Hoàn thiện `mapped_keys(mapping, nums)` — trả về danh sách khóa theo đúng thứ tự đầu vào, dùng **một** `str.maketrans`.

**Bài 4 (tự viết lại):** Viết lại cách sắp bằng **sắp xếp đếm theo cơ số (radix sort LSD)** trên khóa (khóa `< 10⁹` ⇒ 9 lượt chữ số, mỗi lượt sắp đếm **ổn định**). So với `Solution`.

**Bài 5 (mini challenge):** Cho `nums` và một **thứ tự ưu tiên chữ số** `rank` (hoán vị `0..9`), sắp `nums` theo kiểu **từ điển** của dãy chữ số với bảng thứ tự mới (số ngắn hơn là tiền tố thì đứng trước), ổn định. Kiểm bằng `cmp_to_key` viết tay.

In [22]:
# Bài 3 — TODO (chỉ định nghĩa, không gọi)
def mapped_keys(mapping, nums):
    # TODO: table = str.maketrans(...); [int(str(x).translate(table)) for x in nums]
    pass

# Bài 4 — TODO
def jumbled_radix(mapping, nums):
    # TODO: tính khóa; 9 lượt counting sort ổn định theo chữ số (đơn vị -> cao)
    pass

# Bài 5 — TODO
def lexi_custom(rank, nums):
    # TODO: key = tuple(rank[int(c)] for c in str(x))
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1**
- Mức 1: `d → 9 − d`.
- Mức 2: Khóa: `0→9, 1→8, 2→7, 10→89, 99→0`.
- Mức 3: `[99, 2, 1, 0, 10]`.

**Bài 2**
- Mức 1: `1 → 0`, `0 → 3`, `2 → 1`, `3 → 2`.
- Mức 2: `100 → 033 = 33`, `3 → 2`, `20 → 13`.
- Mức 3: `0 → 3`.

**Bài 3**
- Mức 1: `str.maketrans("0123456789", "".join(map(str, mapping)))`.
- Mức 2: `int(...)` bỏ số 0 đầu.
- Mức 3: So với `mapped_arith`.

**Bài 4**
- Mức 1: Khóa `< 10⁹` ⇒ tối đa 9 chữ số.
- Mức 2: Mỗi lượt: 10 xô theo chữ số thứ `k` của **khóa**, giữ thứ tự trong xô.
- Mức 3: LSD radix ổn định ⇒ kết quả = sắp ổn định theo khóa.

**Bài 5**
- Mức 1: Khóa tuple so từ điển như chuỗi.
- Mức 2: Tuple ngắn hơn là tiền tố ⇒ nhỏ hơn (đúng yêu cầu).
- Mức 3: `cmp` viết tay: so từng vị trí theo `rank`, rồi so độ dài.

## 19. Đáp án tham khảo cho bài tập

In [23]:
# Bài 1
r1 = Solution().sortJumbled([9, 8, 7, 6, 5, 4, 3, 2, 1, 0], [0, 1, 2, 10, 99])
print("Bài 1:", r1); assert r1 == [99, 2, 1, 0, 10]

# Bài 2
mp2 = [3, 0, 1, 2, 4, 5, 6, 7, 8, 9]
r2 = [mapped_arith(mp2, x) for x in [100, 3, 20, 0]]
print("Bài 2:", r2); assert r2 == [33, 2, 13, 3]

# Bài 3
def mapped_keys_sol(mapping, nums):
    table = str.maketrans("0123456789", "".join(map(str, mapping)))
    return [int(str(x).translate(table)) for x in nums]

rnd = random.Random(3)
for _ in range(500):
    mp = list(range(10)); rnd.shuffle(mp)
    a = [rnd.randrange(10**rnd.randint(1, 9)) for _ in range(10)]
    assert mapped_keys_sol(mp, a) == [mapped_arith(mp, x) for x in a]
print("Bài 3 OK")

Bài 1: [99, 2, 1, 0, 10]
Bài 2: [33, 2, 13, 3]
Bài 3 OK


In [24]:
# Bài 4
def jumbled_radix_sol(mapping, nums):
    keys = [mapped_arith(mapping, x) for x in nums]
    order = list(range(len(nums)))
    p = 1
    for _ in range(10):                       # khóa < 10^10 (an toàn)
        buckets = [[] for _ in range(10)]
        for i in order:
            buckets[keys[i] // p % 10].append(i)
        order = [i for b in buckets for i in b]
        p *= 10
    return [nums[i] for i in order]

# Bài 5
def lexi_custom_sol(rank, nums):
    return sorted(nums, key=lambda x: tuple(rank[int(c)] for c in str(x)))

def lexi_cmp(rank):
    def cmp(a, b):
        sa, sb = str(a), str(b)
        for ca, cb in zip(sa, sb):
            if rank[int(ca)] != rank[int(cb)]:
                return rank[int(ca)] - rank[int(cb)]
        return len(sa) - len(sb)
    return cmp

rnd = random.Random(5)
for _ in range(500):
    mp = list(range(10)); rnd.shuffle(mp)
    a = [rnd.choice([0, rnd.randrange(100), rnd.randrange(10**9)]) for _ in range(rnd.randint(1, 30))]
    assert jumbled_radix_sol(mp, a) == Solution().sortJumbled(mp, a)
    assert lexi_custom_sol(mp, a) == sorted(a, key=cmp_to_key(lexi_cmp(mp)))
print("Bài 4, Bài 5 OK; ví dụ từ điển tùy biến:", lexi_custom_sol([9, 8, 7, 6, 5, 4, 3, 2, 1, 0], [1, 10, 9, 91, 100]))

Bài 4, Bài 5 OK; ví dụ từ điển tùy biến: [9, 91, 1, 10, 100]


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template:**

```python
def mapped(x):
    if x == 0: return mapping[0]
    res, p = 0, 1
    while x:
        res += mapping[x % 10] * p; p *= 10; x //= 10
    return res
return sorted(nums, key=mapped)
```

**Một dòng:** `t = str.maketrans("0123456789", "".join(map(str, mapping))); sorted(nums, key=lambda x: int(str(x).translate(t)))`.

**Checklist:**
- [ ] `x = 0` ⇒ `mapping[0]`?
- [ ] Đổi chữ số **đồng thời** (không `replace` dây chuyền)?
- [ ] So **giá trị số**, không so chuỗi?
- [ ] Trả về giá trị **gốc**?
- [ ] Không phá hòa theo giá trị (giữ thứ tự đầu vào)?
- [ ] Chữ ký đúng: `def sortJumbled(self, mapping: List[int], nums: List[int]) -> List[int]`?